In [1]:
import os
import datetime
import argparse
from pathlib import Path
import multiprocessing as mp
from functools import partial
import yaml
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from src.module.utils import (
    parse_args,
    load_config,
    load_npy,
    save_npy,
    multiprocess_function,
    select_strategy,
)
from src.module.prepare_dataset import preprocess, prepare

In [2]:
def load_h5(file_path) -> tuple:
    with h5py.File(file_path, "r") as f:
        ts = f["signal/ts"][:]
        sig = f["signal/abp"][:]
        mbp = f["signal/mbp"][:]
        sqi = f["signal/sqi"][:]
        fs = f["signal"].attrs["fs"]
        sig_len = f["signal"].attrs["len"]
        cid = f.attrs["caseid"]

    return cid, fs, sig_len, ts, sig, mbp, sqi


def prepare_cohort(file_path) -> pd.DataFrame:
    cohort = pd.read_csv(file_path).reset_index(drop=True)
    cohort["age_num"] = np.where((cohort["age"] == ">89"), 90, cohort["age"]).astype(
        np.float32  # age < 1
    )
    # VitalDB는 Cardiac & Obstetric surgery는 애초에 배제하고 수집되었다.
    cohort = (
        cohort.query("18 <= age_num")
        .drop_duplicates(["caseid"], keep=False)
        .query("department not in ['Gynecology']")
        .query("ane_type == 'General'")
    )
    return cohort

In [3]:
config = load_config("../../config/conf.yaml")
data_path = Path(config["data_path"]).expanduser()
src_path = data_path / "vitaldb" / "02_04.processed"
cohort_path = data_path / "vitaldb" / "clinical_data.csv"
file_path = sorted(src_path.rglob(f"*{config['fs']}fs*processed.h5"))
strategy = select_strategy("hypophetversion2")
cohort = prepare_cohort(cohort_path)

HypophetVersion2 strategy selected.


In [4]:
cohort_caseid = cohort.caseid.apply(lambda x: str(x).zfill(4)).values
directories = np.array(os.listdir(src_path))
cohort_dirs = directories[np.isin(directories, cohort_caseid)]
cohort_file_path = list(
    filter(lambda x: str(x).split("/")[-2] in cohort_dirs, file_path)
)

In [5]:
input_len = config["fs"] * config["input_len"]
base_name = f"{str(cohort_file_path[0])[:-3]}_{strategy.name}_adj{config['adjacency']}"

In [10]:
cid, _, _, ts, sig, _, _ = load_h5(cohort_file_path[0])
print(cid)

0001


In [13]:
print(f"{base_name}_neg_segments.npy")

/Users/david/Documents/Github/hypotension-classifier/data/vitaldb/02_04.processed/0001/0001_100fs_processed_hypophetversion2_adj20_neg_segments.npy


In [14]:
np.load(f"{base_name}_neg_segments.npy").shape

(10, 3000)